> **InferenceBase's dilemma:** The platform team is spending $80,000/month on OpenAI API calls. The CEO wants to know: can we self-host Llama-3-8B for $15k/month instead? To answer that, the Platform Engineer needs to know which GPU to order — and that requires understanding what a GPU actually does, why its specs translate to LLM workloads the way they do, and what the real bottleneck is for inference.
>
> **The numbers:** Llama-3-8B has 8 billion parameters. At bf16 precision, that's 16 GB of model weights alone — before activations or the KV cache. An RTX 4090 has 24 GB VRAM. An A100 has 80 GB. Does that mean A100 is just "more GPU"? No — the architecture differences run much deeper than capacity.


# GPU Hardware Foundations: Why GPUs Accelerate AI

| Part | Concept                      | Why you need this NOW                                   |
| ---- | ---------------------------- | ------------------------------------------------------- |
| 1    | CPU vs GPU: SIMD vs SIMT     | Why does the same matmul run 10-50× faster on GPU?      |
| 2    | Memory hierarchy             | Why is LLM inference memory-bound, not compute-bound?   |
| 3    | Roofline model               | Which GPU spec matters most for which workload?         |
| 4    | Warp execution and occupancy | Why does batch size affect GPU efficiency non-linearly? |
| 5    | Memory coalescing            | Why does tensor layout affect throughput by 10×?        |
| 6    | Toy → real bridge            | Which GPU should InferenceBase buy, and why?            |

---

> **Prerequisites:** `learning/genai/01-rnns` (PyTorch basics). GPU knowledge: none assumed.
> **Running example:** A `(B=8, S=128, D=256)` attention-shaped matrix multiply — representative of one head in a Transformer layer.


In [ ]:
import subprocess, sys

# Install any of these three packages that aren't already available
for pkg in ["torch", "numpy", "matplotlib"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import torch
import numpy as np
import matplotlib.pyplot as plt
import time

#  GPU detection
# Use CUDA if a GPU is visible to PyTorch, otherwise fall back to CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HAS_GPU = torch.cuda.is_available()

print(f"Device: {DEVICE}")

# Print real hardware specs when a GPU is present, otherwise note we're using reference numbers
if HAS_GPU:
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}")
    print(f"  VRAM:        {props.total_memory / 1e9:.1f} GB")
    print(f"  CUDA cores:  {props.multi_processor_count} SMs")
    print(f"  Compute:     {props.major}.{props.minor}")
else:
    print("No GPU available — showing reference numbers from benchmark literature")
    print("All timing cells still run on CPU; ratios come from published benchmarks")

print()

#  Running example dimensions
B, S, D = 8, 128, 256  # batch=8, seq=128, dim=256 (one attention head)
print(f"Running example: (B={B}, S={S}, D={D}) attention-shaped matmul")
print(f"  This represents 1 of 32 heads in a typical 8B model at S={S}")


---

## Part 1 — CPU vs GPU: Why GPUs Are Fast

A CPU has 4–16 powerful cores optimised for low-latency serial work (branching code, OS tasks, database queries). A GPU has 5,000–16,000 simple cores optimised for high-throughput parallel work.

The difference is **SIMD vs. SIMT**:

- CPU SIMD: 1 instruction, 8-16 data elements at once (AVX-512)
- GPU SIMT: 1 instruction, 32 "threads" execute together (a **warp**)

For matrix multiplication: every output element is an independent dot product. 1,000 independent dot products → 1,000 GPU warps running truly simultaneously.

#### #### Predict first

For a `(512, 512) × (512, 512)` matrix multiply:

How much faster is the GPU compared to a single CPU core?

1. **(a) 2–5×** — GPUs are faster but not dramatically
2. **(b) 10–50×** — GPU's parallelism gives a major speedup on this regular computation
3. **(c) 100–500×** — GPUs are orders of magnitude faster for any matrix operation


> **Intuition first — the warp model:** Imagine 32 factory workers on an assembly line. A foreman shouts one instruction — "tighten bolt" — and all 32 workers execute it simultaneously on their own workpiece. That's a GPU warp: 32 threads, one instruction, maximum parallelism. The GPU's power comes from having thousands of these 32-worker teams all working in parallel. The catch: if even one worker must do something different (branch divergence), the whole team stalls while that worker finishes. Transformer matmuls avoid this almost entirely — pure arithmetic, no branching.


In [ ]:
#  Part 1: CPU vs GPU timing on the running example
Q = torch.randn(B, S, D)  # Query matrix (running example)
K = torch.randn(B, S, D)  # Key matrix


# Time a batched Q@K^T matmul, synchronizing around the GPU launch so timing reflects real execution
def time_matmul(q, k, n_runs=20):

    # Warm up
    for _ in range(3):
        _ = torch.matmul(q, k.transpose(-2, -1))
    if q.is_cuda:
        torch.cuda.synchronize()
    times = []
    for _ in range(n_runs):

        # Sync before/after so CUDA's async queue doesn't hide real kernel time
        if q.is_cuda:
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        result = torch.matmul(q, k.transpose(-2, -1))  # (B, S, S) attention scores
        if q.is_cuda:
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return np.median(times) * 1000, result


cpu_ms, _ = time_matmul(Q, K)
print(f"Running example: Q@K^T with shape ({B},{S},{D}) × ({B},{D},{S})")
print(f"  CPU time: {cpu_ms:.3f} ms")

# Only benchmark on GPU if one is available; otherwise fall back to literature reference numbers
if HAS_GPU:
    Q_gpu = Q.to(DEVICE)
    K_gpu = K.to(DEVICE)
    gpu_ms, _ = time_matmul(Q_gpu, K_gpu)
    speedup = cpu_ms / gpu_ms
    print(f"  GPU time: {gpu_ms:.3f} ms")
    print(f"  Speedup:  {speedup:.1f}×")

    # Bucket the measured speedup into the three candidate predictions
    if speedup > 50:
        print("\n→ Prediction (c) confirmed for this GPU")
    elif speedup > 10:
        print(
            "\n→ Prediction (b) confirmed — significant speedup from parallel execution"
        )
    else:
        print(
            f"\n→ Prediction (a) — small GPU (or short sequence length limits parallelism)"
        )
else:

    # Reference numbers from published benchmarks (A100 vs single CPU core)
    print()
    print("Reference speedups (CPU single-core vs A100, from published benchmarks):")
    for size, speedup in [(256, 15), (512, 45), (1024, 120), (2048, 280)]:
        print(f"  ({size},{size}) matmul: ~{speedup}×")
    print(
        "\n→ The speedup grows with matrix size because larger matrices expose more parallelism"
    )
    print("  Prediction (b) for typical inference sizes; (c) at large batch sizes")


In [ ]:
#  Part 1: Speedup vs. matrix size
sizes = [64, 128, 256, 512]
cpu_times, gpu_times = [], []

# Time the same matmul at increasing sizes on both devices
for s in sizes:
    q = torch.randn(4, s, s)
    k = torch.randn(4, s, s)
    t_cpu, _ = time_matmul(q, k, n_runs=10)
    cpu_times.append(t_cpu)
    if HAS_GPU:
        q_g = q.to(DEVICE)
        k_g = k.to(DEVICE)
        t_gpu, _ = time_matmul(q_g, k_g, n_runs=10)
        gpu_times.append(t_gpu)

# Left panel: raw CPU vs GPU timing curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(sizes, cpu_times, "o-", color="steelblue", label="CPU")
if HAS_GPU:
    axes[0].plot(sizes, gpu_times, "s-", color="coral", label="GPU")
axes[0].set_xlabel("Sequence length S")
axes[0].set_ylabel("Time (ms)")
axes[0].set_title("Matmul time vs. sequence length")
axes[0].legend()

# Right panel: the derived speedup curve (measured if a GPU is present, else literature reference)
if HAS_GPU and gpu_times:
    speedups = [c / g for c, g in zip(cpu_times, gpu_times)]
    axes[1].plot(sizes, speedups, "D-", color="mediumseagreen")
    axes[1].set_xlabel("Sequence length S")
    axes[1].set_ylabel("GPU speedup (×)")
    axes[1].set_title("GPU speedup grows with problem size")
else:

    # Reference curve
    ref_speedups = [4, 15, 45, 120]
    axes[1].plot(
        sizes, ref_speedups[: len(sizes)], "D-", color="mediumseagreen", ls="--"
    )
    axes[1].set_xlabel("Sequence length S")
    axes[1].set_ylabel("GPU speedup (×, reference)")
    axes[1].set_title("GPU speedup grows with problem size (reference A100 data)")

plt.suptitle(
    "CPU vs GPU: the matmul speedup grows as the problem gets larger", fontweight="bold"
)
plt.tight_layout()
plt.show()
print(
    "→ Small problems (S=64): GPU overhead can dominate. Large problems (S=2048): GPU wins clearly."
)


#### What just happened — and what's missing

GPU speedup grows with problem size: at S=64 the kernel launch overhead can eat the win; at S=512 the thousands of parallel cores deliver 10–50×. This matches prediction (b) for typical LLM sequence lengths.

**Missing piece:** Parallel arithmetic is only half the battle. Even a GPU running at full speed hits a wall when generating a single token — not because the cores are slow, but because every token requires reading 16 GB of model weights from VRAM. Reading data, not computing, is the bottleneck. That's Part 2's revelation.


#### #### Your turn — speedup vs. problem size

Change `YOUR_S` and predict whether the speedup goes up or down before running the cell.

- At `YOUR_S = 32`: does the GPU still win? (Hint: kernel launch overhead is ~0.1 ms regardless)
- At `YOUR_S = 1024`: does the speedup plateau or keep growing?


In [ ]:
#  #### Your turn — CPU vs GPU speedup at different S
# # CHANGE: try YOUR_S = 32, 64, 256, 512, 1024 — which gives the most/least GPU benefit?
YOUR_S = (
    512  # sequence length — try 32 (tiny), 256 (typical inference), 1024 (long context)
)

Q_yours = torch.randn(B, YOUR_S, D)
K_yours = torch.randn(B, YOUR_S, D)

t_cpu_yours, _ = time_matmul(Q_yours, K_yours)
print(f"Q@K^T with S={YOUR_S}  (B={B}, D={D}):")
print(f"  CPU time: {t_cpu_yours:.3f} ms")

# Only measure a real GPU speedup if one is available; otherwise interpolate from reference data
if HAS_GPU:
    Q_gpu_yours = Q_yours.to(DEVICE)
    K_gpu_yours = K_yours.to(DEVICE)
    t_gpu_yours, _ = time_matmul(Q_gpu_yours, K_gpu_yours)
    speedup_yours = t_cpu_yours / t_gpu_yours
    print(f"  GPU time: {t_gpu_yours:.3f} ms")
    print(f"  Speedup:  {speedup_yours:.1f}×")
    if speedup_yours > 50:
        print(
            f"\n→ S={YOUR_S}: GPU's massive parallelism is fully utilised — deep into prediction (c)"
        )
    elif speedup_yours > 10:
        print(f"\n→ S={YOUR_S}: GPU wins significantly — prediction (b) confirmed")
    else:
        print(f"\n→ S={YOUR_S}: small problem; kernel launch overhead still visible")
else:
    ref = {32: 4, 64: 8, 128: 15, 256: 35, 512: 60, 1024: 110}

    # Find the closest benchmarked sequence length to interpolate a speedup estimate
    nearest = min(ref.keys(), key=lambda k: abs(k - YOUR_S))
    print(f"  Reference speedup (A100 vs single core) at S≈{nearest}: ~{ref[nearest]}×")
print()
print(
    "→ Key insight: larger S → more independent dot-products → more warps active → higher GPU utilisation."
)
print(
    "  Inference with S=128 already benefits strongly; training with S=2048 benefits even more."
)


---

## Part 2 — Memory Hierarchy: Why LLM Inference Is Memory-Bound

The GPU has a layered memory system. From fastest (smallest) to slowest (largest):

| Level              | Size      | Bandwidth    | Latency     |
| ------------------ | --------- | ------------ | ----------- |
| Registers          | 256 KB/SM | ~28 TB/s     | 1 cycle     |
| L1 / Shared memory | 228 KB/SM | ~19 TB/s     | ~5 cycles   |
| L2 cache           | 40 MB     | ~12 TB/s     | ~200 cycles |
| HBM (main VRAM)    | 24–80 GB  | 0.9–3.4 TB/s | ~600 cycles |

For a matmul: both input matrices must be loaded from HBM. The output must be written back. The computation happens in registers. The ratio of computation to memory traffic — **arithmetic intensity** — determines whether we're compute-bound or memory-bound.

For LLM single-token inference: reading 8B weights from HBM to generate 1 token. At 2 TB/s bandwidth, reading 16 GB (bf16 weights) takes **8ms** per token. The GPU's compute capability (~300 TFLOPS) could theoretically do this in 0.1ms. → **Memory bound: bandwidth, not compute, is the bottleneck.**


> **Intuition first:** Think of GPU memory as a restaurant kitchen. Registers are ingredients on the cutting board — tiny supply, grabbed in one hand motion. SRAM/L1 is the kitchen counter — a bit more space, very fast reach. L2 is the prep table across the room — more space, a few extra steps. HBM is the walk-in fridge at the back — enormous capacity, but every trip takes time. When inference loads Llama-3-8B's 16 GB of weights, it's making thousands of trips to the walk-in fridge per generated token. That's why HBM bandwidth is the bottleneck.


#### #### Predict first

Llama-3-8B at bf16 has **16 GB** of model weights. An A100 has **2 TB/s** HBM bandwidth and **77 TFLOPS** of bf16 compute.

If compute were the only limit, one forward pass would take ~0.05 ms. But memory IS the limit.

How long must each token generation take — just to read the 16 GB of model weights from HBM?

1. **(a) < 1 ms** — 2 TB/s is so fast that 16 GB is read almost instantly
2. **(b) ~8 ms** — reading 16 GB at 2,000 GB/s takes precisely this long
3. **(c) > 50 ms** — HBM bandwidth makes inference impractically slow


![GPU memory hierarchy pyramid: HBM (80 GB, 2 TB/s) → L2 (40 MB) → SRAM/shared (228 KB/SM) → registers (fastest)](images/gpu-memory-hierarchy.png)


In [ ]:
#  Part 2: Measure effective memory bandwidth
def measure_bandwidth_gbps(nbytes, device, n_runs=10):
    """Measure achieved memory bandwidth for a copy operation."""
    src = torch.randn(nbytes // 4, dtype=torch.float32).to(device)  # float32 = 4 bytes
    if src.is_cuda:
        torch.cuda.synchronize()
    times = []

    # Repeat the copy to get a stable median timing
    for _ in range(n_runs):
        if src.is_cuda:
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        dst = src.clone()
        if src.is_cuda:
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    ms = np.median(times) * 1000
    gbps = (2 * nbytes / 1e9) / (np.median(times))  # read + write = 2× bytes
    return gbps, ms


print("Memory bandwidth benchmarks:")

# Sweep copy sizes from small (fits in cache) to large (HBM-bound) on both devices
for mb in [16, 64, 256, 1024]:  # megabytes
    nbytes = mb * 1024 * 1024
    bw_cpu, ms_cpu = measure_bandwidth_gbps(nbytes, torch.device("cpu"))
    print(f"  {mb:4d} MB copy  CPU: {bw_cpu:.1f} GB/s ({ms_cpu:.1f}ms)", end="")
    if HAS_GPU:
        bw_gpu, ms_gpu = measure_bandwidth_gbps(nbytes, DEVICE)
        print(f"  GPU: {bw_gpu:.1f} GB/s ({ms_gpu:.1f}ms)")
    else:
        print(f"  GPU: ~2,000 GB/s (A100 reference)")

print()
print("LLM inference memory-bound calculation:")
model_gb = 16.0  # Llama-3-8B at bf16 = 8B params × 2 bytes
ref_bw = 2000  # GB/s (A100 HBM)

# Time to stream the full model once from HBM sets the floor for per-token latency
token_latency_ms = (model_gb / ref_bw) * 1000
print(f"  Llama-3-8B weights: {model_gb:.0f} GB at bf16")
print(f"  A100 HBM bandwidth: {ref_bw} GB/s")
print(f"  Min latency per token (memory-bound): {token_latency_ms:.1f}ms")
print(
    f"  → At best: {1000/token_latency_ms:.0f} tokens/second (memory bandwidth limit)"
)


#### What just happened — and what's missing

Memory bandwidth is the bottleneck for LLM inference: reading 16 GB of model weights from HBM takes ~8ms on an A100, regardless of compute capability. This is why buying a GPU with higher bandwidth (A100: 2 TB/s) matters more than higher TFLOPS for single-token inference.

**Missing piece:** How do we compare different GPUs systematically? We need a model that captures both compute AND memory constraints — the **roofline model** does exactly this.


#### #### Your turn — memory-bound latency at different model sizes

Change `YOUR_PARAMS_B` (model size) and `YOUR_BYTES` (precision) to see how the bandwidth floor shifts.

- Try `YOUR_PARAMS_B = 7` (Llama-3-8B's smaller sibling) — is it real-time capable?
- Try `YOUR_BYTES = 1` (int8 quantization) — by exactly how much does latency improve?


In [ ]:
#  #### Your turn — memory-bound token latency
# # CHANGE: try different model sizes — how does latency scale with parameter count?
YOUR_PARAMS_B = 70  # billions of parameters — try 7, 70, 405
YOUR_BYTES = 2  # bytes per parameter: fp32=4, bf16=2, int8=1

# Convert parameter count + precision into total model size on disk/in VRAM
model_gb_yours = YOUR_PARAMS_B * 1e9 * YOUR_BYTES / 1e9
ref_bw_gbs = 2000  # A100 HBM bandwidth (GB/s)

# Streaming the whole model once from HBM sets the minimum time per generated token
latency_ms = (model_gb_yours / ref_bw_gbs) * 1000
max_toks_s = 1000 / latency_ms

print(
    f"Model: {YOUR_PARAMS_B}B params  |  Precision: {YOUR_BYTES}-byte ({['','','int8','','fp32','','','bf16'][min(YOUR_BYTES,7)]} equivalent)"
)
print(f"  Model size on disk:          {model_gb_yours:.1f} GB")
print(f"  A100 HBM bandwidth:          {ref_bw_gbs} GB/s")
print(
    f"  Minimum token latency:       {latency_ms:.1f} ms   ← cannot go lower with this hardware"
)
print(f"  Maximum tokens/sec:          {max_toks_s:.0f}")
print()

# Bucket the latency into rough usability tiers
if latency_ms < 5:
    print(
        "→ < 5 ms/token: real-time chat is possible (< 200 ms for a 40-token response)"
    )
elif latency_ms < 30:
    print("→ < 30 ms/token: usable but noticeable; quantization helps")
else:
    print("→ > 30 ms/token: batching and quantization are essential for usable latency")
print()
print(
    "→ Change YOUR_BYTES from 2 (bf16) to 1 (int8): latency halves. That's quantization in one line."
)
print(
    "→ Latency scales LINEARLY with model size — the memory hierarchy sets the floor, not the ceiling."
)


---

## Part 3 — The Roofline Model: Which Spec Matters for Your Workload?

The roofline model predicts performance based on **arithmetic intensity** (AI): FLOP per byte of memory traffic.

$$\text{Performance} = \min(\text{peak TFLOPS},\ \text{bandwidth} \times \text{AI})$$

- If AI < ridge point → **memory-bound**: buy more bandwidth (HBM)
- If AI > ridge point → **compute-bound**: buy more TFLOPS

The **ridge point** is where the two limits intersect: $\text{ridge} = \frac{\text{peak TFLOPS}}{\text{bandwidth (TFLOP/s/TB/s)}}$. A GPU with 77 TFLOPS and 2 TB/s has ridge ≈ 38.5 FLOP/byte.

**LLM workloads and their arithmetic intensity:**

- Single-token decode: AI ≈ 1–5 (very memory-bound — one token reads all weights once)
- Prefill (large prompt): AI ≈ 10–100 (more compute-bound as sequence grows)
- Training: AI ≈ 50–200 (compute-bound with large batch; multiple passes over weights)


> **Intuition first:** Picture a factory with one brilliant machinist (GPU compute) and one slow delivery truck (memory bandwidth). If the machinist finishes processing each batch _before_ the next truck arrives, the machinist sits idle — the truck is the bottleneck. If trucks are piling up waiting for the machinist, the machinist is the bottleneck. Arithmetic intensity (FLOP/byte) measures how much machining you do per truck delivery. LLM inference at 1 token at a time has very low AI — the machinist is nearly always idle waiting for the weight-truck. That's memory-bound.


![Roofline model: LLM decode sits far left in the memory-bound region; matmul sits near the compute ceiling](images/roofline-model.png)


#### #### Predict first

The A100 has **77 TFLOPS** (bf16) and **2 TB/s** HBM bandwidth.  
Ridge point: 77,000 GFLOPS ÷ 2,000 GB/s = **38.5 FLOP/byte**.

LLM single-token decode has arithmetic intensity ≈ **2 FLOP/byte** (far left of the ridge point).

For LLM decode, how much of the A100's 77 TFLOPS compute is actually being used?

1. **(a) Near 100%** — tensor cores are the bottleneck; bandwidth is fine
2. **(b) ~5%** — arithmetic intensity of 2 means only a tiny fraction of compute is reached
3. **(c) 0%** — TFLOPS is completely irrelevant; only bandwidth matters


In [ ]:
#  Part 3: Roofline model
# GPU specs (reference values from published datasheets)
# cost_mo = $/hr (multiply by 730 hr/month to get monthly cost)
gpus = {
    "RTX 4090": {
        "tflops": 165.0,
        "bandwidth_tbs": 1.0,
        "vram_gb": 24,
        "cost_mo": 1.5,
    },  # $/hr
    "A10G": {
        "tflops": 31.2,
        "bandwidth_tbs": 0.6,
        "vram_gb": 24,
        "cost_mo": 3.0,
    },  # $/hr
    "A100 80G": {
        "tflops": 77.0,
        "bandwidth_tbs": 2.0,
        "vram_gb": 80,
        "cost_mo": 10.0,
    },  # $/hr
    "H100": {
        "tflops": 204.0,
        "bandwidth_tbs": 3.35,
        "vram_gb": 80,
        "cost_mo": 25.0,
    },  # $/hr
}

fig, ax = plt.subplots(figsize=(11, 6))
colors = ["coral", "steelblue", "mediumseagreen", "orange"]
ai_range = np.logspace(-1, 3, 300)

# Draw each GPU's roofline: compute ceiling (flat) capped by bandwidth ceiling (diagonal)
for (name, spec), color in zip(gpus.items(), colors):
    bw_gbs = spec["bandwidth_tbs"] * 1000
    ridge = spec["tflops"] * 1000 / bw_gbs  # GFLOP/GB = FLOP/byte
    perf = np.minimum(spec["tflops"] * np.ones_like(ai_range), bw_gbs / 1000 * ai_range)
    ax.loglog(
        ai_range, perf, lw=2, color=color, label=f"{name} (${spec['cost_mo']}/hr)"
    )
    ax.axvline(ridge, color=color, ls=":", lw=0.8, alpha=0.6)

# Annotate key LLM workloads
for label, ai, y_pos in [
    ("LLM decode\n(1 token)", 2, 3),
    ("LLM prefill\n(512 tokens)", 50, 6),
    ("Training\n(batch=32)", 150, 30),
]:
    ax.scatter([ai], [y_pos], s=100, zorder=5, color="black")
    ax.annotate(
        label, (ai, y_pos), textcoords="offset points", xytext=(8, 5), fontsize=8
    )

ax.set_xlabel("Arithmetic Intensity (FLOP/byte)", fontsize=11)
ax.set_ylabel("Attainable Performance (TFLOPS)", fontsize=11)
ax.set_title(
    "Roofline Model — memory-bound left of ridge, compute-bound right", fontsize=12
)
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

print("Key insight: LLM decode (AI≈2) is deep in the memory-bound region.")
print("For InferenceBase's use case: HBM bandwidth matters more than raw TFLOPS.")
print()
print("Best value for Llama-3-8B inference:")

# Rank each GPU by tokens/second per dollar for this specific model size
for name, spec in gpus.items():
    tokens_per_s = (
        spec["bandwidth_tbs"] * 1000 / (16.0 / 1000)
    )  # 16GB model / bandwidth
    efficiency = tokens_per_s / spec["cost_mo"]
    print(
        f"  {name}: ~{tokens_per_s:.0f} tok/s, ${spec['cost_mo']}/hr → {efficiency:.0f} tok/s/dollar"
    )

print("\n→ InferenceBase workload (B=8, S=128, D=256):")
print("   Arithmetic intensity ≈ 0.5 FLOP/byte — memory-bound region of roofline")
print("   This means: bandwidth, not compute, is the bottleneck for LLM inference")
print(
    "   Choice implication: RTX 4090 (936 GB/s) beats A100-SXM (2 TB/s / $10/hr) on tok/s per dollar"
)


#### What just happened — and what's missing

The roofline confirms prediction (b): LLM decode at AI ≈ 2 uses only ~5% of the A100's 77 TFLOPS peak. The entire compute budget is wasted while the GPU waits for 16 GB of weights to arrive over the memory bus. **Bandwidth per dollar, not TFLOPS per dollar, is the right metric for inference GPU shopping.**

**Missing piece:** Even knowing we're bandwidth-limited, the GPU isn't automatically well-utilized. Launch a single-token decode with batch=1 and thousands of warps sit completely idle — not because bandwidth is saturated, but because there simply isn't enough parallel work. Batch size is the control knob, and Part 4 shows why it's the highest-ROI software change InferenceBase can make before buying any new hardware.


#### #### Your turn — explore the roofline for different workloads

Change `YOUR_AI` to see where different LLM workloads land relative to each GPU's ridge point.

- `YOUR_AI = 1`: extreme single-token decode — where on the roofline?
- `YOUR_AI = 50`: prefill with a long prompt — still memory-bound or switching?
- `YOUR_AI = 200`: large-batch training — which GPUs flip to compute-bound?


In [ ]:
#  #### Your turn — roofline at different arithmetic intensities
# # CHANGE: try YOUR_AI = 1 (extreme decode), 50 (prefill), 200 (large-batch training)
YOUR_AI = 50  # FLOP/byte arithmetic intensity

print(f"Attainable throughput at arithmetic intensity = {YOUR_AI} FLOP/byte:")
print(
    f"{'GPU':15s}  {'BW ceiling (TF)':16s}  {'Compute ceil (TF)':18s}  {'Achieved (TF)':14s}  {'Bound by'}"
)
print("-" * 78)

# Compare each GPU's bandwidth-limited vs compute-limited ceiling at this arithmetic intensity
for name, spec in gpus.items():
    bw_gbs = spec["bandwidth_tbs"] * 1000
    bw_ceil = bw_gbs / 1000 * YOUR_AI  # TFLOPS limited by bandwidth
    comp_ceil = spec["tflops"]  # TFLOPS limited by compute
    achieved = min(bw_ceil, comp_ceil)
    bound = "MEMORY" if bw_ceil < comp_ceil else "COMPUTE"
    print(
        f"  {name:13s}  {bw_ceil:10.1f}          {comp_ceil:10.1f}          {achieved:8.1f}        {bound}"
    )

# The ridge point is where the bandwidth and compute ceilings cross for the A100
ridge_a100 = (
    gpus["A100 80G"]["tflops"] * 1000 / (gpus["A100 80G"]["bandwidth_tbs"] * 1000)
)
print()
print(f"A100 ridge point: {ridge_a100:.1f} FLOP/byte")
print(
    f"Your AI ({YOUR_AI}) vs ridge ({ridge_a100:.1f}): {'MEMORY-bound' if YOUR_AI < ridge_a100 else 'COMPUTE-bound'}"
)
print()
print(
    "→ Set YOUR_AI = 1: ALL GPUs deeply memory-bound — TFLOPS spec is nearly irrelevant"
)
print("→ Set YOUR_AI = 200 (large-batch training): GPUs flip to compute-bound")
print(
    "→ The ridge point is the crossover — buying TFLOPS only helps if YOUR_AI exceeds it"
)


---

## Part 4 — Warp Execution and Occupancy

A GPU **warp** is 32 threads that execute the same instruction simultaneously (SIMT).

- **High occupancy:** many active warps → GPU hides memory latency by switching to a ready warp while another waits for HBM
- **Low occupancy:** few active warps → GPU sits idle waiting for memory fetches

For LLM inference with `batch_size=1`: one token, one forward pass, few active warps → low GPU utilization. With `batch_size=32`: 32 independent decode paths → many warps → high utilization.

This explains a counter-intuitive result: **batching amortizes weight reads**. The 16 GB of Llama-3-8B model weights must be read from HBM on every forward pass. With batch=1, those 16 GB generate 1 output token. With batch=32, the same 16 GB generate 32 output tokens — the weight-read cost is shared.

$$\text{throughput} = \frac{\text{bandwidth (GB/s)}}{\text{model size (GB)}} \times \text{batch\_size}$$

Until the KV cache fills VRAM, throughput scales linearly with batch size.


> **InferenceBase context:** Parts 1-3 told you which GPU to rent. Part 4 explains why your code might not fully utilise that GPU — even on the right hardware, inefficient kernels leave 40-60% of FLOPS unused. Every pattern here has a direct cost for InferenceBase.
>
> Recall the running example from Part 1: `(B=8, S=128, D=256)`. Computing `Q @ K^T` for that shape produces `B×S×S = 8×128×128 = 131,072` output entries — roughly 4,096 warps launched per attention head, per forward pass. If InferenceBase's serving code lets any of those warps diverge (e.g. branching on padded tokens), it directly inflates the $/token cost the roofline chart was trying to minimize.


**What the diagram shows:** Teal threads follow the `if` branch; grey threads are idle (waiting for teal to finish before the warp can resume). When half your warp idles, you lose half your throughput — even though the GPU is "busy."


![Warp divergence: 20 threads take the "if" path (teal), 12 are disabled (gray), serializing execution and doubling latency](images/warp-simt-execution.png)


#### #### Predict first

Llama-3-8B decode on an A100 (2 TB/s). At batch=1, throughput is approximately 125 tokens/sec.

Going from `batch_size=1` to `batch_size=32`, how does **total tokens/second** change?

1. **(a) No change** — HBM bandwidth is already saturated at batch=1; extra batch items don't help
2. **(b) ~4–8× faster** — batching helps but with significant diminishing returns early on
3. **(c) ~32× faster** — throughput scales nearly linearly with batch size (until VRAM fills up)


In [ ]:
#  Part 4: Batch size effect on throughput
# Simulate how throughput (tokens/sec) scales with batch size
# For memory-bound inference: throughput scales with batch size until VRAM fills up

D_MODEL = 4096  # Llama-3-8B hidden dim (same 4096 as 7B family)
FF_DIM = 16384  # feedforward expansion (4× hidden)
batch_sizes = [1, 2, 4, 8, 16, 32]


def flops_per_forward(batch, seq=1, d=D_MODEL, ff=FF_DIM):
    """Rough FLOP count for one forward pass (attention + FFN)."""
    attn_flops = 4 * batch * seq * d * d  # QKV projections + output
    ffn_flops = 2 * batch * seq * d * ff  # two linear layers
    return (attn_flops + ffn_flops) / 1e9  # GFLOP


# Memory traffic stays roughly constant (model weights dominate for single-token decode)
model_gb = 16.0  # Llama-3-8B at bf16 = 8B × 2 bytes
ref_bw = 2.0  # TB/s (A100)

print("Throughput analysis for Llama-3-8B single-token decode:")
print(
    f"{'Batch':>6}  {'GFLOPs/step':>12}  {'Arith. Int.':>12}  {'Approx. tok/s':>14}  {'tok/s/req':>10}"
)
print("-" * 65)

# Estimate throughput at each batch size from its arithmetic intensity
for b in batch_sizes:
    flops = flops_per_forward(b)
    mem_gb = model_gb + b * D_MODEL * 2 / 1e9  # model + activations (small)
    ai = flops / mem_gb  # FLOP/GB (arithmetic intensity)
    toks_per_s = ref_bw * 1000 * ai  # bandwidth × AI (memory-bound estimate)
    print(
        f"  {b:4d}  {flops:12.1f}  {ai:12.2f}  {min(toks_per_s, 77000):14.0f}  {min(toks_per_s,77000)/b:10.0f}"
    )

print()
print(
    "→ With batch=1: most of the GPU's 5000+ cores sit idle (low arithmetic intensity)"
)
print(
    "→ With batch=32: more work per memory read → higher GPU utilization → more tok/s/dollar"
)

# Recompute the same throughput curve for plotting
toks_list = []
for b in batch_sizes:
    flops = flops_per_forward(b)
    mem_gb = model_gb + b * D_MODEL * 2 / 1e9
    ai = flops / mem_gb
    toks_list.append(min(ref_bw * 1000 * ai, 77000))

# Left: absolute throughput grows with batch; right: per-request throughput stays flat
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(batch_sizes, toks_list, "o-", color="coral", lw=2)
axes[0].set_xlabel("Batch size")
axes[0].set_ylabel("Tokens/second")
axes[0].set_title("Throughput scales with batch size (memory-bound regime)")

axes[1].plot(
    batch_sizes,
    [t / b for t, b in zip(toks_list, batch_sizes)],
    "s-",
    color="steelblue",
    lw=2,
)
axes[1].set_xlabel("Batch size")
axes[1].set_ylabel("Tokens/second/request")
axes[1].set_title("Per-request latency stays roughly constant until VRAM fills")

plt.suptitle(
    "Batching amortizes model weight reads → linear throughput scaling",
    fontweight="bold",
)
plt.tight_layout()
plt.show()


#### What just happened — and what's missing

Throughput scales nearly linearly with batch size — prediction (c). At batch=1 we read 16 GB of weights to produce 1 token (arithmetic intensity ≈ 0.5 FLOP/byte). At batch=32, the same 16 GB produces 32 tokens (AI ≈ 16). The operating point moves rightward on the roofline from Part 3 — same bandwidth cost, 32× the output.

**Implication:** A batch=1 inference server uses roughly 3% of the GPU's theoretical throughput. Running a dynamic batching server (vLLM, TGI) at batch=32+ recovers ~30× of that "wasted" bandwidth before touching any hardware.

**Missing piece:** We've assumed each byte read from HBM is useful. But _how_ the GPU reads those bytes matters — strided, non-contiguous memory access causes cache-line waste that can throw away 80–97% of bandwidth on each read. That's memory coalescing, and it's Part 5.


#### #### Your turn — find the batch size ceiling

Change `YOUR_MAX_BATCH` and watch where VRAM runs out ( OOM).

- At what batch size does the RTX 4090's 24 GB VRAM overflow?
- What does the `tok/s` column tell you about the cost of hitting that ceiling?


In [ ]:
#  #### Your turn — batch size ceiling on a 24 GB GPU
# # CHANGE: increase YOUR_MAX_BATCH until you hit the VRAM ceiling
YOUR_MAX_BATCH = 64  # try: 32, 64, 128, 256 — at what batch does VRAM run out?

VRAM_GB = 24.0  # RTX 4090 total VRAM
model_b_gb = 16.0  # Llama-3-8B at bf16

# Only keep candidate batch sizes up to the chosen ceiling
your_batches = [b for b in [1, 2, 4, 8, 16, 32, 64, 128, 256] if b <= YOUR_MAX_BATCH]
ref_bw_tb = 2.0  # A100 TB/s (used for throughput estimate)

print(
    f"Throughput up to batch={YOUR_MAX_BATCH}  (VRAM limit: {VRAM_GB} GB on RTX 4090):"
)
print(f"{'Batch':>6}  {'VRAM used':>10}  {'Fits?':>7}  {'Approx. tok/s':>14}")
print("-" * 46)

# Check whether each batch size fits in VRAM and estimate its throughput if it does
for b in your_batches:
    flops_b = flops_per_forward(b)
    vram_used = model_b_gb + b * D_MODEL * 2 / 1e9
    fits = "" if vram_used < VRAM_GB else " OOM"
    ai_b = flops_b / vram_used
    toks_b = min(ref_bw_tb * 1000 * ai_b, 77000) if "" in fits else 0
    print(f"  {b:4d}  {vram_used:8.2f} GB  {fits:8s}  {toks_b:13.0f}")

print()
print("→ Throughput scales linearly until VRAM fills. In practice the KV cache is also")
print(
    "  growing per token (Part 5's problem), so the real ceiling hits sooner than shown here."
)
print(
    "→ Dynamic batching servers (vLLM, TGI) use paged KV cache to push this ceiling higher."
)


---

## Part 5 — Memory Coalescing: Why Tensor Layout Matters

When 32 threads in a warp access memory simultaneously, the GPU can combine them into one or a few memory transactions — if and only if the addresses are **contiguous**. This is **coalesced access**.

**Row-major tensor (PyTorch default):**

- Reading a row = contiguous addresses → 1 memory transaction → coalesced 
- Reading a column = strided addresses (every `N` bytes apart) → 1 transaction per thread → 32× the traffic 

This is why transposing before matmul matters. `A.t()` in PyTorch creates a **non-contiguous view** — the data is unchanged in memory, but the stride metadata flips. When a matmul kernel then reads rows of `A.t()`, those rows are columns of `A` in memory → strided → slow.

**In Transformers specifically:** `K.transpose(-2, -1)` in `Q @ K^T` creates a non-contiguous key matrix every forward pass. FlashAttention avoids this by keeping K tiles in SRAM (shared memory) and reordering compute to avoid the strided HBM access entirely.


> **InferenceBase — Part 5:** Warp divergence (Part 4) wasted compute cycles. Memory coalescing (Part 5) wastes bandwidth cycles. For LLM inference at InferenceBase's scale, DRAM bandwidth is the primary bottleneck — this is why the roofline chart mattered.
>
> The running example `(B=8, S=128, D=256)` illustrates this directly: `K.transpose(-2, -1)` inside `Q @ K^T` turns the `(8, 128, 256)` key tensor into a non-contiguous `(8, 256, 128)` view, forcing strided HBM reads on every one of InferenceBase's forward passes — the same $80k/month bottleneck the roofline model flagged in Part 3.


#### #### Predict first

A `(4096, 4096)` matrix multiply in two forms:

- **Coalesced:** `A @ B` — rows of A are adjacent in memory; 32 warp threads read one cache line
- **Strided:** `A.t() @ B` — `A.t()` is a non-contiguous view; reading its "rows" jumps every 4096 floats

How much slower is the strided form on GPU?

1. **(a) Same speed** — GPU drivers handle non-contiguous layouts transparently at no cost
2. **(b) 20–50% slower** — measurable but moderate; cuBLAS partially compensates
3. **(c) 2–5× slower** — strided HBM access breaks coalescing and wastes the majority of bandwidth


In [ ]:
#  Part 5: Coalesced vs. strided memory access
M = 4096
A = torch.randn(M, M).to(DEVICE)
B_mat = torch.randn(M, M).to(DEVICE)


# Generic timing helper: run fn() n times with GPU sync around each call, return median ms
def time_op(fn, n=10):
    if HAS_GPU:
        torch.cuda.synchronize()
    times = []
    for _ in range(n):
        if HAS_GPU:
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        fn()
        if HAS_GPU:
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return np.median(times) * 1000


# Contiguous matmul (A @ B: rows of A are contiguous in memory)
t_contig = time_op(lambda: torch.matmul(A, B_mat))

# Non-contiguous matmul (A.t() is a strided view: columns of A = strided access)
t_strided = time_op(lambda: torch.matmul(A.t(), B_mat))

# Contiguous after explicit copy (A.t().contiguous() materialises the transposed layout)
A_t_c = A.t().contiguous()
t_contiguous_copy = time_op(lambda: torch.matmul(A_t_c, B_mat))

print(f"Matrix multiply on ({M},{M}) square matrix:")
print(f"  A @ B (contiguous):               {t_contig:.2f} ms  ← baseline")
print(
    f"  A.t() @ B (non-contiguous/strided): {t_strided:.2f} ms  ({t_strided/t_contig:.1f}× slower)"
)
print(
    f"  A.t().contiguous() @ B:             {t_contiguous_copy:.2f} ms  (pre-copy restores speed)"
)
print()
print("→ Non-contiguous tensor operations break memory coalescing.")
print("  In attention: K.transpose(-2,-1) creates a non-contiguous view.")
print("  FlashAttention (covered in learning/ai-infrastructure/03-flash-attention/)")
print("  avoids this by keeping KV in SRAM tiles, sidestepping the strided HBM access.")

# Check contiguity
print()
print(f"  A.is_contiguous():       {A.is_contiguous()}")
print(f"  A.t().is_contiguous():   {A.t().is_contiguous()}")
print(f"  A.t().contiguous().is_contiguous(): {A.t().contiguous().is_contiguous()}")


#### What just happened — and what's missing

A single `.t()` — a zero-copy view operation — slows a 4096×4096 matmul by 2–5×. The reason: 32 warp threads now read from 32 separate 128-byte cache lines instead of one. Each cache line transfer moves 128 bytes but only 4 bytes are used → 97% of bandwidth is wasted filling cache lines that only contribute one useful float.

At InferenceBase's scale, every `K.transpose(-2, -1)` inside the attention loop pays this tax on every token.

**The five concepts now close the loop:**

- Part 1 → GPU runs matmuls 10–50× faster than CPU via massive parallelism
- Part 2 → LLM inference is memory-bound; weight reads, not compute, set the speed limit
- Part 3 → Roofline quantifies the exact split: AI < 38 FLOP/byte → buy bandwidth
- Part 4 → Batching amortizes weight reads → nearly linear throughput scaling
- Part 5 → Coalesced access prevents bandwidth waste _within_ each read

**Part 6 applies all five to answer the original question.**


#### #### Your turn — coalescing penalty vs. matrix size

Change `YOUR_SIZE` and observe whether the penalty grows or shrinks as matrices get larger.

- At `YOUR_SIZE = 512`: is the penalty as bad? (Hint: does the matrix fit in L2?)
- At `YOUR_SIZE = 4096`: this matches Llama-3-8B's hidden dim — what's the real-world penalty?


In [ ]:
#  #### Your turn — coalescing penalty at different matrix sizes
# # CHANGE: try YOUR_SIZE = 512, 1024, 2048, 4096 — does the penalty grow or shrink?
YOUR_SIZE = 2048  # matrix dimension (N × N)

A_yours = torch.randn(YOUR_SIZE, YOUR_SIZE).to(DEVICE)
B_yours = torch.randn(YOUR_SIZE, YOUR_SIZE).to(DEVICE)

# Time the same contiguous / strided / copy-first matmul pattern at this matrix size
t_coal = time_op(lambda: torch.matmul(A_yours, B_yours))
t_strid = time_op(lambda: torch.matmul(A_yours.t(), B_yours))
t_copy = time_op(lambda: torch.matmul(A_yours.t().contiguous(), B_yours))

penalty = t_strid / t_coal

print(
    f"Matrix size: {YOUR_SIZE} × {YOUR_SIZE}  (rows/cols = {YOUR_SIZE} floats = {YOUR_SIZE*4} bytes)"
)
print(f"  Coalesced   A @ B:                 {t_coal:.2f} ms  ← baseline")
print(f"  Strided     A.t() @ B:             {t_strid:.2f} ms  ({penalty:.2f}× slower)")
print(f"  Copy-first  A.t().contiguous() @ B:{t_copy:.2f} ms")
print()

# Bucket the measured penalty into rough severity tiers
if penalty < 1.3:
    print("→ Penalty is small — matrix fits in GPU L2 cache; stride is mostly hidden")
elif penalty < 2.5:
    print(
        "→ Moderate penalty — strided access is costing you ~50% bandwidth efficiency"
    )
else:
    print(
        "→ Large penalty — non-contiguous layout wastes majority of HBM bandwidth per read"
    )
print()
print("→ At YOUR_SIZE=4096 (close to LLM hidden dim) the penalty is most severe,")
print(
    "  because the full 4096-element stride exceeds the L2 cache line and forces HBM reads."
)
print(
    "→ FlashAttention solves this by tiling K into SRAM — the strided K view never touches HBM."
)


---

## Part 6 — Toy → Real: Which GPU Should InferenceBase Buy?

Let's apply everything we've learned to answer the original question: which GPU for Llama-3-8B at < $15k/month?

**Checklist so far:**

| Requirement                                 | Comes from                                  |
| ------------------------------------------- | ------------------------------------------- |
| VRAM > 16 GB (model) + ~4 GB (KV cache)     | Part 2 — memory hierarchy                   |
| Buy bandwidth, not TFLOPS                   | Part 3 — roofline model (AI ≈ 2 for decode) |
| Use batch_size > 1 to amortize weight reads | Part 4 — warp occupancy                     |
| Keep KV tensors contiguous in memory        | Part 5 — memory coalescing                  |

The GPU table below compares four candidates against these requirements.


In [ ]:
#  Part 6: GPU selection for InferenceBase
model_gb_bf16 = 16.0  # Llama-3-8B at bf16

print("GPU selection analysis for Llama-3-8B inference:")
print(
    f"{'GPU':15s} {'VRAM':6s} {'BW TB/s':8s} {'Fits?':6s} {'Tok/s':7s} {'$/hr':6s} {'Tok/$/hr':8s}"
)
print("-" * 65)

# Check VRAM fit and estimate throughput/cost-efficiency for every candidate GPU
for name, spec in gpus.items():
    fits = "" if spec["vram_gb"] > model_gb_bf16 + 4 else ""  # +4GB for KV cache
    bw_gbs = spec["bandwidth_tbs"] * 1000
    toks = bw_gbs / (model_gb_bf16 / 1000)
    tpd = toks / spec["cost_mo"]
    print(
        f"  {name:13s} {spec['vram_gb']:4.0f}GB  {spec['bandwidth_tbs']:6.1f}    {fits:5s}  {toks:6.0f}  {spec['cost_mo']:5.2f}  {tpd:8.0f}"
    )

print()
print("InferenceBase analysis:")
print(f"  Budget target:  < $15,000/month")
print(
    f"  Required VRAM:  > {model_gb_bf16:.0f} GB (model) + 4 GB (KV cache) = 20 GB minimum"
)
print()

# Best value: RTX 4090
rtx4090 = gpus["RTX 4090"]
bw_4090 = rtx4090["bandwidth_tbs"] * 1000
toks_4090 = bw_4090 / (model_gb_bf16 / 1000)
monthly_cost_4090 = rtx4090["cost_mo"] * 730  # $/hr × 730 hr/month

print(f"  RECOMMENDATION: RTX 4090")
print(
    f"    VRAM: 24 GB    (fits with {24 - model_gb_bf16 - 4:.0f} GB headroom for KV cache)"
)
print(f"    Throughput: ~{toks_4090:.0f} tok/s (bandwidth-limited)")
print(
    f"    Monthly cost: ~${monthly_cost_4090:,.0f} (730 hr/month × ${rtx4090['cost_mo']}/hr)"
)
print(f"    Savings vs. OpenAI: ~${80000 - monthly_cost_4090:,.0f}/month")
print()
a100 = gpus["A100 80G"]
monthly_cost_a100 = a100["cost_mo"] * 730  # $/hr × 730 hr/month
print(f"  WHY NOT A100?")
print(
    f"    A100 has {a100['bandwidth_tbs']/rtx4090['bandwidth_tbs']:.1f}× more bandwidth ({a100['bandwidth_tbs']:.1f} vs {rtx4090['bandwidth_tbs']:.1f} TB/s)"
)
print(f"    → {a100['bandwidth_tbs']/rtx4090['bandwidth_tbs']:.1f}× more tokens/second")
print(
    f"    But costs {a100['cost_mo']/rtx4090['cost_mo']:.0f}× more per hour (${a100['cost_mo']} vs ${rtx4090['cost_mo']})"
)

# A100 costs: cost_mo ($/hr) × 730 = monthly cost
print(
    f"  A100 80G:   {gpus['A100 80G']['vram_gb']:.0f}GB  {gpus['A100 80G']['bandwidth_tbs']:.1f} TB/s  ${gpus['A100 80G']['cost_mo']*730:,.0f}/mo"
)
print(
    f"    → Fits budget, but {gpus['A100 80G']['cost_mo']/gpus['RTX 4090']['cost_mo']:.0f}× hourly cost for only {gpus['A100 80G']['bandwidth_tbs']/gpus['RTX 4090']['bandwidth_tbs']:.1f}× more bandwidth"
)
print(f"    → RTX 4090 wins on tok/s per dollar for inference workloads")


---

## Summary and Closing Decision

| Part | Concept           | Key insight                                                                   |
| ---- | ----------------- | ----------------------------------------------------------------------------- |
| 1    | CPU vs GPU        | GPUs win on large parallel matmuls (10–100×); CPU wins on serial branchy code |
| 2    | Memory hierarchy  | LLM inference is memory-bound: HBM bandwidth, not TFLOPS, is the bottleneck   |
| 3    | Roofline model    | AI < ridge point → buy bandwidth; AI > ridge → buy TFLOPS                     |
| 4    | Warp occupancy    | batch=1 underutilizes GPU; batch=32 amortizes model weight reads              |
| 5    | Memory coalescing | Strided access (non-contiguous tensors) can be 2–5× slower than coalesced     |
| 6    | GPU selection     | RTX 4090 for inference: best tok/s/dollar for memory-bound workloads          |

### Key insights to keep

- **Bandwidth > TFLOPS for LLM inference.** At AI ≈ 2 FLOP/byte, the A100's 77 TFLOPS peak is used at ~5%. You pay for 95% of those tensor cores and don't use them during decode.
- **The same GPU is a bandwidth machine (inference) and a compute machine (training).** The roofline ridge point — ~38 FLOP/byte on an A100 — marks the exact crossover. Buy bandwidth below it; buy TFLOPS above it.
- **Batching is a free 30× throughput multiplier.** Serving requests at batch=1 wastes 97% of possible bandwidth efficiency. Dynamic batching (vLLM, TGI) is the highest-ROI optimization before any hardware change.
- **Memory coalescing is a silent tax.** A single `.t()` without `.contiguous()` can halve effective HBM bandwidth by wasting 97% of each cache-line transfer. FlashAttention's entire design exists to avoid this.
- **VRAM capacity gates what fits; bandwidth gates how fast.** An A100 (80 GB / 2 TB/s) and RTX 4090 (24 GB / 1 TB/s) are both capable of running Llama-3-8B — but the right choice depends on whether model capacity or throughput is the binding constraint.
- **For InferenceBase:** RTX 4090 at ~$1,095/month saves ~$79k vs. OpenAI at identical quality. The hardware knowledge paid for itself before the first GPU shipped.


In [ ]:
#  Closing Decision
print("=" * 60)
print("  CLOSING DECISION — InferenceBase GPU Selection")
print("=" * 60)
print()
print("  Llama-3-8B inference requirements:")
print(f"    Model size (bf16): {model_gb_bf16:.0f} GB")
print(f"    Workload type: memory-bound (AI ≈ 2–5 for single-token decode)")
print()
print("  SELECTED: RTX 4090 @ ~$1.50/hr")
print(f"    VRAM: 24 GB   (fits with {24 - model_gb_bf16 - 4:.0f} GB for KV cache)")
print(f"    Throughput: ~{toks_4090:.0f} tok/s (BW-limited)")
print(f"    Monthly cost: ~${monthly_cost_4090:,.0f}")
print(f"    Savings vs. OpenAI: ~${80000 - monthly_cost_4090:,.0f}/month")
print()
print("  WHY NOT A100?")
print(f"    A100 has 3.3× more bandwidth (2.0 vs 1.0 TB/s) → ~3.3× faster")
print(
    f"    But costs {gpus['A100 80G']['cost_mo']/rtx4090['cost_mo']:.0f}× more per hour"
)
print(f"    For inference: RTX 4090 wins on tok/s per dollar")
print()
print("  The GPU spec that matters most for LLM inference: HBM BANDWIDTH")
print("  (not TFLOPS, not core count, not clock speed)")

---

## What This Notebook Covered (and What It Didn't)

### Tier 1 — Implemented and Demonstrated

- **CPU vs GPU timing** — matmul speedup measured across sequence lengths
- **Memory bandwidth** — measured on actual hardware (or reference numbers on CPU)
- **Roofline model** — arithmetic intensity computed for LLM workloads; plotted for 4 GPUs
- **Batch size and occupancy** — throughput-per-dollar analysis across batch sizes
- **Memory coalescing** — contiguous vs. strided matmul timing
- **GPU selection** — concrete recommendation for InferenceBase with measured justification

### Tier 2 — Explained but Not Built

- **Tensor Cores** — specialised matrix multiply units (A100: 312 TFLOPS bf16 peak); the roofline model assumes them; building a custom Tensor Core kernel is out of scope here

### Tier 3 — Named but Out of Scope

- **NVLink / NVSwitch** — high-speed GPU interconnect for multi-GPU systems; matters for training, less for single-GPU inference
- **MIG (Multi-Instance GPU)** — partition one A100 into up to 7 independent GPU instances; useful for multi-tenant serving
- **Triton custom kernels** — writing custom GPU kernels in Python; covered in `learning/ai-infrastructure/08-triton-kernels/`


---

## When to Use What

| Workload                      | Bottleneck      | Buy                    | Avoid                |
| ----------------------------- | --------------- | ---------------------- | -------------------- |
| LLM single-token decode       | HBM bandwidth   | More GB/s (A100, H100) | Pure TFLOPS upgrades |
| LLM prefill (large prompt)    | Mixed           | Balanced (H100)        | —                    |
| Training, large batch         | Compute         | More TFLOPS (H100 FP8) | Bandwidth-only GPUs  |
| Serving multiple small models | Memory capacity | More VRAM              | —                    |

→ **Next:** `learning/ai-infrastructure/02-mixed-precision/` — now that you understand the memory hierarchy, this chapter answers: how do precision formats (fp32, fp16, bf16, int8) affect how much model fits in VRAM, and what breaks when you reduce precision?
